# All layers: does adapting the whole model help?

Every adapter so far applies LoRA to the last 16 layers (mlx-lm's default), which is 16 of 24 in Qwen2.5-0.5B and 16 of 28 in the 1.5B. "LoRA Without Regret" (Thinking Machines, 2025) reports that LoRA on all layers does better than on a subset. Here the same recipe is trained with LoRA on every layer (`--num-layers -1`) for the 0.5B and the 1.5B, and compared with the 16-layer runs of ModelSize.ipynb, which used exactly the same settings on the same machine (NAS GPU, micro-batches of 1 with `--grad-accum 4`, gradient checkpointing).

It also tests a confound of the model-size ladder: the bigger the model, the smaller the share of it that 16 layers cover.

## Prediction (written before running)

I'm not sure. I know that moving more parameters doesn't help by itself (LoraRank.ipynb). Touching every layer for something specific is fine as long as you don't break, or don't mind losing, the model's old memory, and part of that memory is how to process language. Adapting the early layers might give more weight to the words I use most, but touching the whole model might also need more data, which brings me back to the theory that data is what's missing.

How it will be read: if data is the limit, all layers lands within ±0.02 of the 16-layer run and reaches its minimum earlier (more freedom, same data). If the early layers help with my vocabulary, the gain shows up mostly on short replies, where word choice is most of the reply.

In [1]:
import json, os, subprocess
from pathlib import Path
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
ROOT = Path("..").resolve()
R = ROOT / "experiments" / "results"
PY = os.environ.get("TLM_PY", "uv run python")              # NAS: TLM_PY=/opt/venv/bin/python
TURNS_V1 = "data/processed_v1_2026-10-17/turns.jsonl"
# size label, model, all-layers adapter, 16-layer adapter from ModelSize.ipynb, layers in the model
MODELS = [
    ("0.5B", "mlx-community/Qwen2.5-0.5B-Instruct-4bit", "alllayers_0.5b", "size_0.5b", 24),
    ("1.5B", "mlx-community/Qwen2.5-1.5B-Instruct-4bit", "alllayers_1.5b", "size_1.5b", 28),
]

def run(cmd):
    """Run a command from the repo root, show its output here, and stop the notebook if it fails."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if "Fetching" not in line and "█" not in line:
            print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"failed ({p.returncode}): {cmd}")

def best(name):
    d = json.load(open(R / f"checkpoints_{name}.json"))
    v = {int(k): x for k, x in d["valid"].items()}
    s = min(v, key=v.get)
    return s, v[s]

## Train with LoRA on every layer, pick the checkpoint on validation, measure the signal

In [2]:
for label, model, name, _, _ in MODELS:
    if not (ROOT / "adapters" / name / "0003500_adapters.safetensors").exists():
        run(f"{PY} scripts/train.py --name {name} --data data/dataset/session --model {model} --num-layers -1 --iters 3500 --grad-accum 4 --grad-checkpoint")
    run(f"{PY} scripts/eval_checkpoints.py adapters/{name} --model {model} --data data/dataset/session/valid.jsonl --steps 500:3500:500 --out experiments/results/checkpoints_{name}.json")
    step, loss = best(name)
    print(f"{label} all layers: best step {step}, valid {loss:.3f}")
    run(f"{PY} scripts/eval_signal.py --adapter adapters/{name} --step {step} --model {model} --turns {TURNS_V1} --out experiments/results/signal_{name}.json")

$ /opt/venv/bin/python scripts/train.py --name alllayers_0.5b --data data/dataset/session --model mlx-community/Qwen2.5-0.5B-Instruct-4bit --num-layers -1 --iters 3500 --grad-accum 4 --grad-checkpoint
5404 train examples, batch 4: 3500 steps (2.59 epochs)
/opt/venv/bin/python -m mlx_lm lora --model mlx-community/Qwen2.5-0.5B-Instruct-4bit --train --data data/dataset/session --mask-prompt --adapter-path /work/talk-like-me/.nasrun/snap/alllayers/adapters/alllayers_0.5b --iters 14000 --batch-size 1 --learning-rate 1e-05 --num-layers -1 --max-seq-length 2048 --save-every 400 --steps-per-eval 400 --val-batches 25 --seed 0 --grad-accumulation-steps 4 --grad-checkpoint
Loading pretrained model
Loading datasets
Training
╭─ mlx_lm.lora ────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│    model           mlx-community/Qwen2.5-0.5B-Instruct-4bit                                                          │
│    type            lora · -1 layer

## Summary: all layers against the last 16

In [3]:
print(f"{'model':>5} {'layers':>10} {'best step':>9} {'best valid':>10} {'signal':>8} {'95% CI':>14}")
for label, model, name, name16, layers in MODELS:
    for n, lay in [(name16, f"16 of {layers}"), (name, f"{layers} of {layers}")]:
        step, loss = best(n)
        sig = json.load(open(R / (f"signal_size_{label}.json" if n == name16 else f"signal_{n}.json")))
        print(f"{label:>5} {lay:>10} {step:>9} {loss:>10.3f} {sig['signal'] * 100:>7.1f} [{sig['signal_ci'][0] * 100:.1f}, {sig['signal_ci'][1] * 100:.1f}]")

model     layers best step best valid   signal         95% CI
 0.5B   16 of 24      2500      3.424    21.2 [15.4, 27.6]
 0.5B   24 of 24      2500      3.381    20.8 [15.5, 26.7]
 1.5B   16 of 28      2500      3.147    21.4 [15.3, 28.3]
 1.5B   28 of 28      2500      3.108    22.1 [16.3, 28.6]


## Paired difference and where the gain is

Both adapters are scored on the same validation replies, so the difference can be paired by session. Split by reply length (reply tokens) to test the vocabulary reading of the prediction: if the early layers help with my words, the gain should be largest on short replies.

In [ ]:
# Paired difference all layers - 16 layers on the same validation replies, by session; and where the gain is, by reply length.
import json
import sys
from pathlib import Path
sys.path.insert(0, str(ROOT))
from talk_like_me.sessions import valid_sessions
from talk_like_me.stats import paired_diff

R = ROOT / "experiments" / "results"
sessions = valid_sessions()
BINS = [(1, 5), (6, 15), (16, 40), (41, 10**9)]
for label, all_name, n16 in [("0.5B", "alllayers_0.5b", "size_0.5b"), ("1.5B", "alllayers_1.5b", "size_1.5b")]:
    a = json.load(open(R / f"checkpoints_{n16}.json"))["per_example"]["2500"]
    b = json.load(open(R / f"checkpoints_{all_name}.json"))["per_example"]["2500"]
    assert len(a) == len(b) == len(sessions)
    d, ci = paired_diff(a, b, sessions)
    print(f"{label}: all layers - 16 layers = {d:+.3f} nats [{ci[0]:+.3f}, {ci[1]:+.3f}] (negative = all layers better)")
    for lo, hi in BINS:
        idx = [i for i, (_, n) in enumerate(a) if lo <= n <= hi]
        da = sum(a[i][0] for i in idx) / sum(a[i][1] for i in idx)
        db = sum(b[i][0] for i in idx) / sum(b[i][1] for i in idx)
        tok = sum(a[i][1] for i in idx)
        print(f"   replies of {lo}-{hi if hi < 10**9 else '...'} tokens ({len(idx)} replies, {tok} tokens): {db - da:+.3f}")


0.5B: all layers - 16 layers = -0.043 nats [-0.068, -0.025] (negative = all layers better)
   replies of 1-5 tokens (163 replies, 687 tokens): -0.016
   replies of 6-15 tokens (464 replies, 4640 tokens): -0.046
   replies of 16-40 tokens (318 replies, 7588 tokens): -0.042
   replies of 41-... tokens (67 replies, 4411 tokens): -0.045
1.5B: all layers - 16 layers = -0.039 nats [-0.054, -0.028] (negative = all layers better)
   replies of 1-5 tokens (163 replies, 687 tokens): -0.051
   replies of 6-15 tokens (464 replies, 4640 tokens): -0.059
   replies of 16-40 tokens (318 replies, 7588 tokens): -0.033
   replies of 41-... tokens (67 replies, 4411 tokens): -0.027


## Conclusion

**Prediction:** unsure. If data is the limit, all layers lands within ±0.02 of the 16-layer run and reaches its minimum earlier; if the early layers help with my vocabulary, the gain shows up mostly on short replies.

**Neither reading holds: all layers is clearly better, and evenly.**
- **Validation loss improves by about 0.04 nats on both models**, outside the ±0.02 band: 0.5B 3.424 → 3.381 (paired -0.043 [-0.068, -0.025]), 1.5B 3.147 → 3.108 (-0.039 [-0.054, -0.028]). Both intervals exclude 0. The 0.5B with all layers (3.381) is the best 0.5B adapter so far, below the ~3.40 where the 16-layer runs stopped improving.
- **The minimum is not earlier**: step 2,500 for all four runs. More freedom didn't make it overfit sooner, which is what "data is the limit" predicted.
- **The gain is not concentrated in short replies**: 0.5B -0.016 / -0.046 / -0.042 / -0.045 nats for replies of 1-5 / 6-15 / 16-40 / 41+ tokens, 1.5B -0.051 / -0.059 / -0.033 / -0.027. Roughly the same everywhere.
- **The me-vs-friends signal doesn't change** (0.5B 21.2 → 20.8, 1.5B 21.4 → 22.1, intervals of ±6). As with model size: adapting more of the model predicts everyone's WhatsApp better, not more specifically me.

**Reading:** the 16-layer default was leaving something on the table, in line with what "LoRA Without Regret" reports. Unlike rank (LoraRank.ipynb: 4 to 32 barely matters), *where* the adapter sits matters more than *how big* it is. This also confounds the model-size ladder a bit: with 16 layers the 1.5B was adapting a smaller share of itself (16 of 28) than the 0.5B (16 of 24).

**Limits.**
- One run per configuration. The paired interval covers which replies were sampled, not training noise: Seeds.ipynb measures how much the best validation loss moves between seeds. If seeds move it by about 0.04 too, this result weakens. **Checked in AllLayersSeeds.ipynb:** with seeds 1 and 2 the 0.5B gain is -0.037 and -0.036, and the largest seed-to-seed spread within an arm is 0.023, so the result holds.
- Same machine and recipe for both arms (NAS, micro-batch 1, `--grad-accum 4`, gradient checkpointing), so they're comparable with each other; not bit for bit with the Mac's run1_5ep.
